# 07a_score_ours — 우리 ModernBERT 라우터 점수 (EV-10)

[최종 실험 계획서](../../../docs/ml/ROUTER_FINAL_EXPERIMENT_PLAN.md) 4.5절. 03(R0-집단·R1-집단·R1-집단-A)과 02(RT-08)의 학습된 모델로 dev·test·RouterArena 질문 전체의 점수를 한 번 계산한다. 03은 자기 검증 행에만 점수를 냈고(R1-집단-A는 A 분할만), 02는 점수를 내지 않아서, 07_lock이 원천마다 보정하려면 이 단계가 필요하다.

- 학습 때와 같은 토큰화(512토큰, 뒤쪽 자름)와 RT-05의 점수 함수를 쓴다. 로컬 시험에서 03의 검증 점수와 차이는 2e-5 이하였다.
- test는 질문 원문 열만 읽는다. 결과는 Drive `scores-modernbert/<모델>/`에 모델마다 저장된다.

**런타임: 2026.07(Python 3.12), T4 GPU.** 예상 약 30분(추정). 04~06과 독립이라 동시에 돌려도 된다. 끊기면 본 실행 셀만 다시 실행하면 끝난 모델은 건너뛴다.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path
import hashlib, json, subprocess, sys, zipfile
RUN = Path('/content/drive/MyDrive/LLM_Router/router-final/runs/run001')
SOURCE_ZIP = RUN / 'code' / 'router-final-source-lock.zip'
expected = SOURCE_ZIP.with_suffix('.sha256').read_text().split()[0]
assert hashlib.sha256(SOURCE_ZIP.read_bytes()).hexdigest() == expected, '소스 ZIP이 바뀌었다'
REPO = Path('/content/router-final-source')
with zipfile.ZipFile(SOURCE_ZIP) as z:
    for name in z.namelist():
        assert (REPO / name).resolve().is_relative_to(REPO.resolve()), name
    z.extractall(REPO)
for name, digest in json.loads((REPO / 'source-bundle-manifest.json').read_text()).items():
    assert hashlib.sha256((REPO / name).read_bytes()).hexdigest() == digest, name
assert sys.version_info[:2] in {(3, 11), (3, 12)}, '런타임 버전을 2026.07(Python 3.12)로 바꾸세요'
subprocess.run([sys.executable, '-m', 'pip', 'install', 'torch==2.5.1', 'torchvision==0.20.1', 'torchaudio==2.5.1',
                '--index-url', 'https://download.pytorch.org/whl/cu124'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(REPO / 'ml/notebooks/router_final/requirements-train-colab.txt')], check=True)

설치가 끝나면 **런타임 → 세션 다시 시작**을 한 번 한 뒤 아래 셀부터 실행합니다.

In [ ]:
from pathlib import Path
import json, os, shutil, subprocess, sys
RUN = Path('/content/drive/MyDrive/LLM_Router/router-final/runs/run001')
REPO = Path('/content/router-final-source')
FINAL = RUN / 'scores-modernbert'
import torch
assert torch.cuda.is_available(), 'T4 GPU 런타임을 선택하세요'
print(torch.cuda.get_device_name(0), torch.__version__)

def run_logged(script, args):
    """Run a script and stream its log into this cell (subprocess.run hides it in Colab)."""
    proc = subprocess.Popen([sys.executable, str(REPO / 'ml/notebooks/router_final' / script), *map(str, args)],
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
                            env=dict(os.environ, PYTHONUNBUFFERED='1'))
    for line in proc.stdout:
        print(line, end='')
    if proc.wait():
        raise RuntimeError(f'실패(exit {proc.returncode}). 위 로그의 마지막 부분(Traceback)을 보내 주세요')

In [ ]:
# 1) 시험 실행: 역할마다 40문항 (몇 분)
shutil.rmtree('/content/scores-smoke', ignore_errors=True)
run_logged('score_modernbert.py', ['--run-dir', RUN, '--output-dir', '/content/scores-smoke', '--smoke', '--device', 'cuda'])

In [ ]:
# 2) 본 실행 → Drive scores-modernbert/ (모델 4개)
run_logged('score_modernbert.py', ['--run-dir', RUN, '--output-dir', FINAL, '--device', 'cuda'])
print(json.loads((FINAL / 'manifest.json').read_text())['outputs'])